In [ ]:
# Célula 0 — Montagem do Google Drive e mudança para o diretório de trabalho
import os
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    Path("/content/drive/MyDrive/coder-comprimido").mkdir(parents=True, exist_ok=True)
except ImportError:
    pass

%cd /content/drive/MyDrive/coder-comprimido

# F0b — Teste de Equivalência Isolado: vLLM Oficial (PR `#60149`) vs `lib/dory.py`

Use este notebook caso queira executar o teste de equivalência em uma sessão limpa do Colab, sem instalar `trl`/`peft` no mesmo ambiente do `vLLM`.

**Critério de aprovação:**
1. Comparação estrutural de `lib/dory.py` contra `hf://models/tiny-random/dory-1.0/vllm_dory.py`.
2. Execução de 5 prompts com `1`, `2` e `4` loops no vLLM oficial e na nossa reimplementação `DoryForCausalLM`, comparando os logprobs dos primeiros 32 tokens gerados (decodificação gulosa, mesma entrada).
3. Se o Top-1 divergir em mais de **2%** (`0.02`) das posições, o teste falha e exibe exatamente onde ocorreu cada divergência.

In [ ]:
# 1. Instalação enxuta apenas do vLLM da PR #60149 e transformers==4.57.6
!VLLM_USE_PRECOMPILED=1 pip install -q "git+https://github.com/vllm-project/vllm.git@refs/pull/60149/head"
!pip install -q "transformers==4.57.6" "accelerate==1.13.0" "safetensors==0.5.3"

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from lib.io import DEFAULT_DRIVE_ROOT, setup_drive_and_cache
PATHS = setup_drive_and_cache(drive_root=DEFAULT_DRIVE_ROOT, local_fallback_root=REPO_ROOT)

In [ ]:
# 2. Comparação com `hf://models/tiny-random/dory-1.0/vllm_dory.py` e Teste Numérico (5 prompts, 1/2/4 loops, 32 tokens)
from huggingface_hub import hf_hub_download
from lib.dory import DEFAULT_EQUIVALENCE_PROMPTS, DORY_HF_REPO, verify_vllm_hf_equivalence

vllm_ref_path = Path(hf_hub_download(repo_id="tiny-random/dory-1.0", filename="vllm_dory.py"))
print("Arquivo de referência oficial baixado em:", vllm_ref_path)

report = verify_vllm_hf_equivalence(
    prompts=DEFAULT_EQUIVALENCE_PROMPTS,
    loops=(1, 2, 4),
    max_new_tokens=32,
    max_divergence_rate=0.02,
    model_name_or_path=DORY_HF_REPO,
    output_json_path=str(PATHS["results"] / "equivalence_vllm_hf.json"),
    halt_on_failure=True,
)